In [1]:
# ================================================================
# FULL NLP + K-MEANS CLUSTERING PROJECT
# Dataset: target_all.csv
# ================================================================

# ================================================================
# 1. IMPORT LIBRARIES
# ================================================================

import os
import re
import string
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score
)

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

print("=" * 70)
print("NLP + K-MEANS CLUSTERING")
print("=" * 70)


# ================================================================
# 2. LOAD DATASET
# ================================================================

FILE_PATH = "target_all.csv"

if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(
        f"Could not find '{FILE_PATH}'. "
        "Make sure the CSV file is in the same folder as this notebook."
    )

df = pd.read_csv(FILE_PATH)

print("\nDataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nFirst 5 rows:")
display(df.head())


# ================================================================
# 3. DATASET INFORMATION
# ================================================================

print("\n" + "=" * 70)
print("DATASET INFORMATION")
print("=" * 70)

print("\nShape:")
print(df.shape)

print("\nColumn names:")
for i, column in enumerate(df.columns, 1):
    print(f"{i}. {column}")

print("\nData types:")
display(df.dtypes.to_frame("Data Type"))

print("\nMissing values:")
missing_values = df.isnull().sum().sort_values(ascending=False)
display(missing_values.to_frame("Missing Values"))

print("\nDuplicate rows:", df.duplicated().sum())


# ================================================================
# 4. REMOVE DUPLICATES
# ================================================================

original_rows = len(df)

df = df.drop_duplicates().reset_index(drop=True)

print(
    f"\nRemoved {original_rows - len(df)} duplicate rows."
)
print("Remaining rows:", len(df))


# ================================================================
# 5. IDENTIFY TEXT COLUMNS
# ================================================================

text_columns = df.select_dtypes(
    include=["object", "string"]
).columns.tolist()

print("\n" + "=" * 70)
print("TEXT COLUMNS")
print("=" * 70)

if len(text_columns) == 0:
    raise ValueError(
        "No text columns were found in the dataset."
    )

for column in text_columns:
    print(f"- {column}")


# ================================================================
# 6. AUTOMATICALLY SELECT TEXT COLUMN
# ================================================================
#
# The code selects the text column with the largest average
# amount of text.
#
# If you already know the correct text column, replace this
# section with:
#
# TEXT_COLUMN = "your_column_name"
#
# ================================================================

text_lengths = {}

for column in text_columns:
    text_lengths[column] = (
        df[column]
        .fillna("")
        .astype(str)
        .str.len()
        .mean()
    )

text_length_table = pd.DataFrame(
    list(text_lengths.items()),
    columns=["Column", "Average Text Length"]
).sort_values(
    "Average Text Length",
    ascending=False
)

print("\nText column statistics:")
display(text_length_table)

TEXT_COLUMN = text_length_table.iloc[0]["Column"]

print(f"\nSelected text column: '{TEXT_COLUMN}'")


# ================================================================
# 7. OPTIONAL: COMBINE MULTIPLE TEXT COLUMNS
# ================================================================
#
# If you want to use multiple text columns, you can replace
# the previous selection with something like:
#
# TEXT_COLUMNS_TO_USE = ["title", "description", "category"]
#
# and combine them.
#
# The default uses the automatically selected column.
# ================================================================

TEXT_COLUMNS_TO_USE = [TEXT_COLUMN]

df["combined_text"] = (
    df[TEXT_COLUMNS_TO_USE]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
)


# ================================================================
# 8. TEXT CLEANING FUNCTION
# ================================================================

def clean_text(text):

    # Convert to string
    text = str(text)

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(
        r"http\S+|www\S+|https\S+",
        " ",
        text
    )

    # Remove email addresses
    text = re.sub(
        r"\S+@\S+",
        " ",
        text
    )

    # Remove HTML tags
    text = re.sub(
        r"<.*?>",
        " ",
        text
    )

    # Remove mentions
    text = re.sub(
        r"@\w+",
        " ",
        text
    )

    # Remove hashtags symbol but keep the word
    text = re.sub(
        r"#(\w+)",
        r"\1",
        text
    )

    # Remove numbers
    text = re.sub(
        r"\d+",
        " ",
        text
    )

    # Remove punctuation
    text = text.translate(
        str.maketrans(
            "",
            "",
            string.punctuation
        )
    )

    # Remove extra whitespace
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    # Strip leading/trailing spaces
    text = text.strip()

    return text


# ================================================================
# 9. APPLY TEXT CLEANING
# ================================================================

df["clean_text"] = (
    df["combined_text"]
    .fillna("")
    .apply(clean_text)
)

print("\nExample of cleaned text:")

display(
    df[
        ["combined_text", "clean_text"]
    ].head(10)
)


# ================================================================
# 10. REMOVE EMPTY TEXT
# ================================================================

before_empty_removal = len(df)

df = df[
    df["clean_text"].str.strip().ne("")
].copy()

df = df.reset_index(drop=True)

removed_empty = (
    before_empty_removal - len(df)
)

print(
    f"\nRemoved {removed_empty} rows with empty text."
)

print(
    "Rows available for clustering:",
    len(df)
)


# ================================================================
# 11. DOWNLOAD / DEFINE STOPWORDS
# ================================================================
#
# Using sklearn's built-in English stopwords avoids requiring
# an external download.
# ================================================================

print("\n" + "=" * 70)
print("NLP PROCESSING")
print("=" * 70)


# ================================================================
# 12. TF-IDF VECTORIZATION
# ================================================================

tfidf = TfidfVectorizer(
    stop_words="english",

    # Use single words and two-word phrases
    ngram_range=(1, 2),

    # Ignore extremely rare words
    min_df=2,

    # Ignore words occurring in almost every document
    max_df=0.95,

    # Maximum number of features
    max_features=10000,

    # Normalize vectors
    sublinear_tf=True
)

X = tfidf.fit_transform(
    df["clean_text"]
)

print("\nTF-IDF completed.")

print(
    "Number of documents:",
    X.shape[0]
)

print(
    "Number of TF-IDF features:",
    X.shape[1]
)

print(
    "TF-IDF matrix shape:",
    X.shape
)


# ================================================================
# 13. DISPLAY TOP TF-IDF WORDS
# ================================================================

feature_names = (
    tfidf.get_feature_names_out()
)

mean_scores = (
    np.asarray(X.mean(axis=0))
    .ravel()
)

top_features = pd.DataFrame({
    "Word": feature_names,
    "TF-IDF Score": mean_scores
})

top_features = (
    top_features
    .sort_values(
        "TF-IDF Score",
        ascending=False
    )
    .head(30)
)

print("\nTop 30 TF-IDF features:")

display(top_features)


# ================================================================
# 14. DETERMINE POSSIBLE RANGE OF K
# ================================================================

n_samples = X.shape[0]

if n_samples < 3:
    raise ValueError(
        "At least 3 non-empty text records are required "
        "for clustering."
    )

# Maximum K should never be larger than number of samples - 1
max_k = min(10, n_samples - 1)

k_values = list(
    range(2, max_k + 1)
)

print("\nK values to test:")
print(k_values)


# ================================================================
# 15. FIND OPTIMAL K USING SILHOUETTE SCORE
# ================================================================

silhouette_scores = []
inertia_values = []
calinski_scores = []
davies_scores = []

print("\n" + "=" * 70)
print("TESTING DIFFERENT K VALUES")
print("=" * 70)

for k in k_values:

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20,
        max_iter=500
    )

    labels = model.fit_predict(X)

    inertia = model.inertia_

    silhouette = silhouette_score(
        X,
        labels
    )

    calinski = calinski_harabasz_score(
        X.toarray(),
        labels
    )

    davies = davies_bouldin_score(
        X.toarray(),
        labels
    )

    inertia_values.append(inertia)
    silhouette_scores.append(silhouette)
    calinski_scores.append(calinski)
    davies_scores.append(davies)

    print(
        f"K={k:2d} | "
        f"Inertia={inertia:.4f} | "
        f"Silhouette={silhouette:.4f} | "
        f"Calinski={calinski:.2f} | "
        f"Davies={davies:.4f}"
    )


# ================================================================
# 16. RESULTS OF K TESTING
# ================================================================

k_results = pd.DataFrame({
    "K": k_values,
    "Inertia": inertia_values,
    "Silhouette Score": silhouette_scores,
    "Calinski-Harabasz Score": calinski_scores,
    "Davies-Bouldin Score": davies_scores
})

print("\nK evaluation results:")

display(k_results)


# ================================================================
# 17. SELECT BEST K
# ================================================================

best_k = k_values[
    np.argmax(silhouette_scores)
]

print(
    "\nBest K based on Silhouette Score:",
    best_k
)

print(
    "Best Silhouette Score:",
    round(
        max(silhouette_scores),
        4
    )
)


# ================================================================
# 18. ELBOW METHOD
# ================================================================

plt.figure(figsize=(9, 6))

plt.plot(
    k_values,
    inertia_values,
    marker="o"
)

plt.xlabel(
    "Number of Clusters (K)"
)

plt.ylabel(
    "Inertia"
)

plt.title(
    "Elbow Method for K-Means"
)

plt.xticks(k_values)

plt.grid(True)

plt.tight_layout()

plt.show()


# ================================================================
# 19. SILHOUETTE SCORE GRAPH
# ================================================================

plt.figure(figsize=(9, 6))

plt.plot(
    k_values,
    silhouette_scores,
    marker="o"
)

plt.xlabel(
    "Number of Clusters (K)"
)

plt.ylabel(
    "Silhouette Score"
)

plt.title(
    "Silhouette Score for Different K Values"
)

plt.xticks(k_values)

plt.grid(True)

plt.tight_layout()

plt.show()


# ================================================================
# 20. FINAL K-MEANS MODEL
# ================================================================

print("\n" + "=" * 70)
print("FINAL K-MEANS MODEL")
print("=" * 70)

kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=20,
    max_iter=500
)

cluster_labels = (
    kmeans.fit_predict(X)
)

df["cluster"] = cluster_labels

print(
    "K-Means clustering completed."
)


# ================================================================
# 21. CLUSTER COUNTS
# ================================================================

cluster_counts = (
    df["cluster"]
    .value_counts()
    .sort_index()
)

cluster_percentages = (
    cluster_counts /
    len(df) * 100
)

cluster_distribution = pd.DataFrame({
    "Cluster": cluster_counts.index,
    "Number of Records": cluster_counts.values,
    "Percentage": cluster_percentages.values
})

cluster_distribution[
    "Percentage"
] = cluster_distribution[
    "Percentage"
].round(2)

print("\nCluster distribution:")

display(cluster_distribution)


# ================================================================
# 22. FINAL CLUSTER EVALUATION
# ================================================================

final_silhouette = silhouette_score(
    X,
    df["cluster"]
)

final_calinski = calinski_harabasz_score(
    X.toarray(),
    df["cluster"]
)

final_davies = davies_bouldin_score(
    X.toarray(),
    df["cluster"]
)

print("\n" + "=" * 70)
print("FINAL CLUSTER EVALUATION")
print("=" * 70)

print(
    f"Silhouette Score:          {final_silhouette:.4f}"
)

print(
    f"Calinski-Harabasz Score:   {final_calinski:.4f}"
)

print(
    f"Davies-Bouldin Score:      {final_davies:.4f}"
)


# ================================================================
# 23. TOP WORDS FOR EACH CLUSTER
# ================================================================

print("\n" + "=" * 70)
print("TOP WORDS FOR EACH CLUSTER")
print("=" * 70)

cluster_keywords = {}

order_centroids = (
    kmeans.cluster_centers_
    .argsort()[:, ::-1]
)

for cluster_number in range(best_k):

    top_indices = (
        order_centroids[
            cluster_number,
            :20
        ]
    )

    top_words = [
        feature_names[index]
        for index in top_indices
    ]

    cluster_keywords[
        cluster_number
    ] = top_words

    print(
        f"\nCluster {cluster_number}:"
    )

    print(
        ", ".join(top_words)
    )


# ================================================================
# 24. CREATE CLUSTER KEYWORD TABLE
# ================================================================

keyword_rows = []

for cluster_number in range(best_k):

    row = {
        "Cluster": cluster_number
    }

    for i, word in enumerate(
        cluster_keywords[cluster_number],
        start=1
    ):
        row[f"Keyword_{i}"] = word

    keyword_rows.append(row)

cluster_keywords_df = pd.DataFrame(
    keyword_rows
)

print("\nCluster keyword table:")

display(cluster_keywords_df)


# ================================================================
# 25. SHOW SAMPLE RECORDS FROM EACH CLUSTER
# ================================================================

print("\n" + "=" * 70)
print("SAMPLE RECORDS FROM EACH CLUSTER")
print("=" * 70)

for cluster_number in range(best_k):

    print(
        f"\n{'=' * 25} "
        f"CLUSTER {cluster_number} "
        f"{'=' * 25}"
    )

    cluster_data = df[
        df["cluster"] == cluster_number
    ]

    sample_size = min(
        10,
        len(cluster_data)
    )

    display(
        cluster_data[
            [TEXT_COLUMN, "cluster"]
        ].head(sample_size)
    )


# ================================================================
# 26. FIND REPRESENTATIVE DOCUMENTS
# ================================================================
#
# Documents closest to each centroid are useful for understanding
# what each cluster represents.
# ================================================================

print("\n" + "=" * 70)
print("REPRESENTATIVE DOCUMENTS")
print("=" * 70)

representative_indices = []

for cluster_number in range(best_k):

    cluster_indices = np.where(
        cluster_labels == cluster_number
    )[0]

    cluster_vectors = X[
        cluster_indices
    ]

    centroid = (
        kmeans.cluster_centers_[
            cluster_number
        ]
    )

    # Euclidean distance
    distances = np.linalg.norm(
        cluster_vectors.toarray() -
        centroid,
        axis=1
    )

    closest_position = np.argmin(
        distances
    )

    original_index = (
        cluster_indices[
            closest_position
        ]
    )

    representative_indices.append(
        original_index
    )

representative_df = df.loc[
    representative_indices,
    [TEXT_COLUMN, "cluster"]
].copy()

representative_df = (
    representative_df
    .sort_values("cluster")
)

display(representative_df)


# ================================================================
# 27. 2D DIMENSIONALITY REDUCTION
# ================================================================
#
# TF-IDF can contain thousands of dimensions.
# TruncatedSVD reduces it to two dimensions for visualization.
# ================================================================

print("\nPerforming dimensionality reduction...")

svd = TruncatedSVD(
    n_components=2,
    random_state=42
)

X_2d = svd.fit_transform(X)

print(
    "Explained variance ratio:",
    svd.explained_variance_ratio_
)

print(
    "Total explained variance:",
    round(
        svd.explained_variance_ratio_.sum(),
        4
    )
)


# ================================================================
# 28. VISUALIZE K-MEANS CLUSTERS
# ================================================================

plt.figure(figsize=(12, 8))

scatter = plt.scatter(
    X_2d[:, 0],
    X_2d[:, 1],
    c=df["cluster"],
    alpha=0.65,
    s=40
)

plt.xlabel(
    "SVD Component 1"
)

plt.ylabel(
    "SVD Component 2"
)

plt.title(
    "NLP + K-Means Clustering"
)

plt.colorbar(
    scatter,
    label="Cluster"
)

plt.grid(True)

plt.tight_layout()

plt.show()


# ================================================================
# 29. TRANSFORM K-MEANS CENTROIDS TO 2D
# ================================================================

centers_2d = svd.transform(
    kmeans.cluster_centers_
)


# ================================================================
# 30. VISUALIZE CLUSTERS + CENTROIDS
# ================================================================

plt.figure(figsize=(12, 8))

plt.scatter(
    X_2d[:, 0],
    X_2d[:, 1],
    c=df["cluster"],
    alpha=0.5,
    s=35
)

plt.scatter(
    centers_2d[:, 0],
    centers_2d[:, 1],
    marker="X",
    s=250,
    edgecolors="black",
    linewidths=1.5
)

for cluster_number in range(best_k):

    plt.annotate(
        f"Cluster {cluster_number}",
        (
            centers_2d[
                cluster_number,
                0
            ],
            centers_2d[
                cluster_number,
                1
            ]
        ),
        xytext=(8, 8),
        textcoords="offset points",
        fontsize=11,
        fontweight="bold"
    )

plt.xlabel(
    "SVD Component 1"
)

plt.ylabel(
    "SVD Component 2"
)

plt.title(
    "K-Means Clusters with Centroids"
)

plt.grid(True)

plt.tight_layout()

plt.show()


# ================================================================
# 31. CREATE FINAL CLUSTER SUMMARY
# ================================================================

summary_rows = []

for cluster_number in range(best_k):

    cluster_data = df[
        df["cluster"] == cluster_number
    ]

    summary_rows.append({
        "Cluster":
            cluster_number,

        "Records":
            len(cluster_data),

        "Percentage":
            round(
                len(cluster_data) /
                len(df) * 100,
                2
            ),

        "Top Keywords":
            ", ".join(
                cluster_keywords[
                    cluster_number
                ][:10]
            )
    })

cluster_summary = pd.DataFrame(
    summary_rows
)

print("\n" + "=" * 70)
print("FINAL CLUSTER SUMMARY")
print("=" * 70)

display(cluster_summary)


# ================================================================
# 32. ADD HUMAN-READABLE CLUSTER LABELS
# ================================================================
#
# K-Means generates numerical labels.
# This creates labels based on the most important words.
# ================================================================

cluster_names = {}

for cluster_number in range(best_k):

    keywords = cluster_keywords[
        cluster_number
    ][:3]

    cluster_names[
        cluster_number
    ] = (
        f"Cluster {cluster_number}: "
        + " / ".join(keywords)
    )

df["cluster_label"] = (
    df["cluster"]
    .map(cluster_names)
)

print("\nCluster labels:")
for cluster, label in cluster_names.items():
    print(f"{cluster} -> {label}")


# ================================================================
# 33. FINAL DATASET PREVIEW
# ================================================================

print("\n" + "=" * 70)
print("FINAL DATASET")
print("=" * 70)

display(
    df.head(20)
)


# ================================================================
# 34. SAVE FULL CLUSTERED DATASET
# ================================================================

OUTPUT_FILE = (
    "target_all_kmeans_results.csv"
)

df.to_csv(
    OUTPUT_FILE,
    index=False
)

print(
    f"\nFull clustered dataset saved as: "
    f"{OUTPUT_FILE}"
)


# ================================================================
# 35. SAVE CLUSTER SUMMARY
# ================================================================

SUMMARY_FILE = (
    "target_all_cluster_summary.csv"
)

cluster_summary.to_csv(
    SUMMARY_FILE,
    index=False
)

print(
    f"Cluster summary saved as: "
    f"{SUMMARY_FILE}"
)


# ================================================================
# 36. SAVE CLUSTER KEYWORDS
# ================================================================

KEYWORDS_FILE = (
    "target_all_cluster_keywords.csv"
)

cluster_keywords_df.to_csv(
    KEYWORDS_FILE,
    index=False
)

print(
    f"Cluster keywords saved as: "
    f"{KEYWORDS_FILE}"
)


# ================================================================
# 37. SAVE K EVALUATION RESULTS
# ================================================================

K_RESULTS_FILE = (
    "target_all_k_evaluation.csv"
)

k_results.to_csv(
    K_RESULTS_FILE,
    index=False
)

print(
    f"K evaluation results saved as: "
    f"{K_RESULTS_FILE}"
)


# ================================================================
# 38. FINAL REPORT
# ================================================================

print("\n")
print("=" * 70)
print("FINAL REPORT")
print("=" * 70)

print(
    f"Original dataset rows:       {original_rows}"
)

print(
    f"Final rows clustered:        {len(df)}"
)

print(
    f"Text column used:            {TEXT_COLUMN}"
)

print(
    f"TF-IDF features:             {X.shape[1]}"
)

print(
    f"Optimal K:                   {best_k}"
)

print(
    f"Silhouette Score:            {final_silhouette:.4f}"
)

print(
    f"Calinski-Harabasz Score:     {final_calinski:.4f}"
)

print(
    f"Davies-Bouldin Score:       {final_davies:.4f}"
)

print("\nCluster sizes:")

for cluster_number in range(best_k):

    count = (
        cluster_counts
        .get(cluster_number, 0)
    )

    percentage = (
        count / len(df) * 100
    )

    print(
        f"  Cluster {cluster_number}: "
        f"{count} records "
        f"({percentage:.2f}%)"
    )

print("\nTop keywords by cluster:")

for cluster_number in range(best_k):

    print(
        f"\nCluster {cluster_number}:"
    )

    print(
        ", ".join(
            cluster_keywords[
                cluster_number
            ][:10]
        )
    )

print("\n" + "=" * 70)
print("PROCESS COMPLETED SUCCESSFULLY")
print("=" * 70)

print("\nGenerated files:")
print("1.", OUTPUT_FILE)
print("2.", SUMMARY_FILE)
print("3.", KEYWORDS_FILE)
print("4.", K_RESULTS_FILE)

NLP + K-MEANS CLUSTERING

Dataset loaded successfully.
Rows: 115
Columns: 3

First 5 rows:


,L,W,D
0,1,0.23,4.5
1,1,0.23,9.0
2,1,0.23,13.5
3,1,0.46,4.5
4,1,0.46,9.0



DATASET INFORMATION

Shape:
(115, 3)

Column names:
1. L
2. W
3. D

Data types:


,Data Type
L,int64
W,float64
D,float64



Missing values:


,Missing Values
L,0
W,0
D,0



Duplicate rows: 0

Removed 0 duplicate rows.
Remaining rows: 115

TEXT COLUMNS


ValueError: No text columns were found in the dataset.